# Windtunnel — how the whole system works

*A high-level tour of the pipeline, the agents, and the moving parts.*

> **An Innovation Month 2026 (IM2026) entry.** Windtunnel is a demonstration
> system built for Innovation Month 2026 — *Ready or Not…*

<img src="frontend/src/img/ReadyOrNotColour.png" alt="Innovation Month 2026 — Ready or Not…" width="320">

**Read this to understand _what the system does and how the pieces fit_ — not what
each line of code does.** It is written for someone new to the repo: a reviewer, a
future maintainer, or a curious public servant. Where it helps, it points you at the
document that owns the detail (`PROJECT_BRIEF.md` for intent, `TECH_SPEC.md` for the
build, `DESIGN_BRIEF.md` for the interface).

---

## In one paragraph

Windtunnel takes a public servant's loose idea for an AI solution and carries it
through two phases. The **Brainstorm phase** is a co-design conversation that sharpens
the idea into a structured outline (optionally with an HTML proof-of-concept and an
information-flow map). The **Governance phase** then stress-tests those artefacts with
a college of specialist AI agents, assessing the concept against the Australian
Government's *AI impact assessment tool* (DTA v1.0) and producing a draft impact
assessment as a Jupyter notebook (the source of record) and an HTML report (the thing
you hand to subject-matter experts). The metaphor is the point: **test the design under
load before you build the aircraft.**

> **The one rule that governs everything below: _models argue, code computes._** AI
> agents may reason about consequences and likelihoods, but they never assign a risk
> rating. Every rating is computed deterministically by plain code from the DTA's own
> risk matrix. This removes the most attackable failure mode of the whole system.

## 1. The shape of the system — three programs and one shared store

Nothing in Windtunnel keeps important state in memory. Everything durable lives in
**one public GitHub repository**, and the three running programs coordinate only
through it — they never share a disk or talk to each other directly.

| Piece | Where it runs | What it does |
| --- | --- | --- |
| **SPA** (React/Vite) | GitHub Pages | The entire user interface. Talks to the backend; polls run status to animate the pipeline. Holds no secrets. |
| **Brainstorm backend** (FastAPI) | **Render** (free tier) | Runs the live, interactive Brainstorm conversation (real-time Gemini calls), issues run codes, commits brainstorm state, and *kicks off* Governance runs. |
| **Governance pipeline** (Python) | GitHub Actions | The multi-agent assessment. Triggered on demand, it commits every checkpoint back to the repo as it goes. |
| **The repository** | Public GitHub repo | The single durable store **and** a public audit trail. Holds the code, the corpus, the built knowledge bases, and every run's state under `runs/<run-id>/`. |

**Why a public repo holds everything.** Public visibility gives the competition a real
audit trail, unlocks unlimited free GitHub Actions minutes on standard runners, and
makes every run commit inspectable. The accepted consequence — everything a user
submits is world-readable — is disclosed up front in the usage warning, which asks
users to keep out anything sensitive, classified, or personally identifying. The
warning makes no claim about what sensitivity the system itself is accredited to
handle; it is purely a public-disclosure notice.

**Why the repo is the *only* store.** Render's disk is ephemeral and Render sleeps when
idle; a GitHub Actions job is a fresh container every time. So the design treats the
repo — not any process's memory — as the system of record. Any program can die and be
restarted, and the run picks up exactly where it left off, because the truth is in git.

### 1.1 How Render is used — the interactive intermediary

Render is the **only always-on server**, and it plays a deliberately narrow role: it
handles the parts that must happen *live, in conversation with a person*, and it is the
**broker** between the browser (which holds no secrets) and GitHub (which needs
credentials to write).

Concretely, the Render backend:

1. **Runs the Brainstorm phase live.** The interviewer, the sufficiency check, the
   feasibility gate, and the PoC / flow-map generators all make real-time Gemini calls
   from Render, because the user is sitting there waiting for each reply.
2. **Issues run codes and creates runs.** It draws a human-readable code (e.g.
   `WT-7K3D-Q2`), checks it's free, and commits the initial `runs/<code>/` skeleton.
3. **Is the sole holder of the secrets.** The Gemini API key and a fine-grained GitHub
   token (PAT) live only in Render's environment variables — **never** in the browser.
   The SPA asks Render to do anything that needs a key; Render does it and returns only
   the result.
4. **Commits brainstorm state to the repo** via the GitHub Contents API (not a local
   `git push` — Render's disk is throwaway), so the browser can rehydrate after a
   refresh and the pipeline can later read the outline.
5. **Triggers the Governance pipeline** with a `workflow_dispatch` call, then steps
   back. It does **not** run the assessment.
6. **Proxies status and downloads.** During a run the SPA polls
   `GET /api/runs/{id}/status`, which Render fetches fresh from GitHub (with ETag
   conditional requests) and passes back. Proxying — rather than reading GitHub's public
   CDN directly — is what keeps the animation near-real-time and keeps any token off the
   browser.

**The cold-start caveat.** Render's free tier sleeps when idle, so the very first call
can take ~60 s to wake. The SPA hides this: it fires a cheap `GET /api/health` the
instant the user passes the warning gate (so the wake overlaps with them reading and
typing), and shows an honest "waking the workshop" state. Once a governance run is
polling, the continuous traffic keeps Render warm, so the cold start effectively only
ever bites once, right at the start.

> **A subtle but important deploy rule:** Render **auto-deploy is off** (or scoped to
> `backend/**`). The pipeline commits to the repo every ~20 s during a run; if Render
> redeployed on every commit it would restart continuously and evict live brainstorm
> sessions.

### 1.2 The end-to-end data flow, in one line

> The SPA drives the **backend** through Brainstorm and commits artefacts under a run
> id → submission tells the backend to **dispatch the Governance workflow** → the
> **pipeline** runs in Actions, committing a checkpoint after every stage and
> regenerating `status.json` on each commit → the SPA **polls `status.json`** (via the
> backend proxy) to animate the pipeline and to detect the two user pauses → on
> completion the notebook and HTML report are committed and downloadable.

```
   Browser (GitHub Pages, no secrets)
        │  REST/JSON
        ▼
   Render backend  ──(Gemini)──►  Brainstorm agents (live)
        │  commits via GitHub API          │
        │  workflow_dispatch               ▼
        ▼                          runs/<id>/brainstorm/*  ─┐
   GitHub Actions ── Governance pipeline ── commits ───────┤
        │  (Gemini + built-in token)                        ▼
        └──────────────────────────────────►  runs/<id>/{run.json, status.json, ...}
                                                        ▲
   Browser polls status ◄── Render status proxy ────────┘
```

## 2. The life of a run — the state machine

Every run is a folder, `runs/<run-id>/`, and the run **code is the id** (no separate
UUID): `WT-7K3D-Q2` normalises straight to `runs/WT-7K3D-Q2/`. That makes resume a
simple path lookup and keeps the public trail human-navigable. The code is a *locator,
not a secret* — the whole repo is public — so it only needs to be unique and easy to
read aloud.

A run walks through a fixed set of **states**. Each state has entry conditions, does its
work, writes its outputs, **commits them as a checkpoint**, and advances. The states:

| State | What happens | Runs on |
| --- | --- | --- |
| `BRAINSTORM` | Interview → outline; optional PoC / flow map | Render |
| `SUBMITTED` | User submits at the gate; backend dispatches the workflow | Render → Actions |
| `THRESHOLD_DRAFTING` | Two generalists draft DTA sections 1–4 **independently, in parallel** | Actions |
| `THRESHOLD_RECONCILING` | Reconciler merges the two drafts; **rating engine computes 3.1–3.9**; routing decided | Actions |
| `THRESHOLD_REVIEW` | **User pause** — review the threshold result, optionally revise (≤2), then route | (paused) |
| `FULL_DRAFTING` | Six specialists retrieve + draft **their own sections only**; may raise ≤3 questions each | Actions |
| `FULL_CHECKPOINT` | **User pause** — *only if* any question was raised; skipped on the happy path | (paused) |
| `FULL_REVISING` | Specialists revise their own sections in light of the answers | Actions |
| `ARCHITECT` | Solution Architect writes the Implementation Plan appendix | Actions |
| `REVIEW` | Reviewer audits coverage + coherence, directs amendments (≤2 cycles); residual 12.3/12.4 computed | Actions |
| `ASSEMBLY` | Notebook + HTML report built and committed | Actions |
| `COMPLETE` | Terminal — report shown. May be revised ≤2 times (`USER_REVISION`). | (terminal) |
| `CONCLUDED` | Terminal — user concluded at threshold because all risks were low | (terminal) |
| `FAILED` | Any unhandled error; run code surfaced; **resumable** from the last checkpoint | (paused) |

**The two user pauses are Actions run-boundaries.** A GitHub Actions job must never sit
idle waiting for a human (it wastes minutes and risks the 6-hour ceiling). So at
`THRESHOLD_REVIEW` and `FULL_CHECKPOINT` the job writes its checkpoint, marks the run
`awaiting_user`, and **exits cleanly**. When the user acts, a brand-new
`workflow_dispatch` resumes the run from where it paused. This is why *a paused run and
a failed run resume by exactly the same mechanism* — both are just a fresh dispatch that
reads `run.json` and continues.

## 3. How each state is saved to the repo

This is the heart of Windtunnel's resilience, so it's worth being precise. Two files
carry a run's state, with deliberately different jobs:

- **`run.json` is authoritative.** It is the state machine: current `stage`,
  `stage_status`, revision counts, review-cycle count, the attestation, and a
  `checkpoints` map recording the commit SHA at which each completed state's outputs
  were durably written. **The pipeline resumes from `run.json` and nothing else.**
- **`status.json` is a *derived projection* for the UI.** It is what the SPA polls to
  animate the graph. It is always safe to recompute from `run.json` plus the committed
  event log, and **the pipeline never resumes from it.**

Everything a state produces is written as **files under `runs/<run-id>/`**, then
committed in one go — that commit *is* the checkpoint. A sketch of the tree:

```
runs/WT-7K3D-Q2/
├─ run.json                 # authoritative state machine
├─ status.json              # derived UI projection
├─ brainstorm/              # outline.md, poc.html, flow-map.mmd/.svg, transcript.jsonl
├─ threshold/               # generalist_a/b.json, reconciled.*, ratings.json, routing.json, divergence.json
├─ full/                    # specialists/<id>.json+.md, questions.json, answers.json,
│                           #   architect.md, reviewer/cycle_N.json, ratings_residual.json
├─ gaps.json                # aggregated "recommended next steps" register
├─ provenance.json          # models, prompt versions, corpus manifests, tokens, attestation
├─ transcripts/<stage>/<agent>.json   # full prompt+response per call (audit)
└─ artefacts/               # assessment.ipynb + assessment.html (the deliverables)
```

**Idempotent resume.** On every dispatch, `run.py`: reads `run.json`; for the target
state, **checks whether its checkpoint outputs already exist and are committed** — if so
it skips past (never redoing work); otherwise it runs the state, commits, and advances.
A job that dies mid-state loses only the uncommitted in-progress work of that *one*
state, which simply re-runs. This means any stage is safe to run twice, which is the
whole resume model.

**One poll fully determines the visible state.** `status.json`'s `nodes` field is a
*whole-graph map* on every write (every node's state, every poll), the event log is
append-only with stable ids, and a `heartbeat` always exists. So the browser can miss
polls and still render the pipeline correctly from a single file.

**No commit races.** Every writer only ever touches its own `runs/<run-id>/…` files
(and ingestion only touches `kb/…`), so two concurrent runs never contend for the same
file. A shared commit helper does *fetch → rebase onto main → push*, retrying on
non-fast-forward; because writers touch disjoint paths, rebases apply cleanly.

## 4. Phase 1 — Brainstorm: the co-design agents

The Brainstorm phase runs **live on Render** and turns a loose idea into a structured
**outline** — the single source of truth for the concept. Everything downstream
regenerates from the outline; artefacts are never patched independently.

The outline is a fixed template of nine sections (`problem`, `solution`,
`users_stakeholders`, `data`, `happy_path`, `alternatives`, `ux_ui`, `constraints`,
`success_criteria`) with machine anchors and YAML front-matter, so downstream agents
parse it deterministically.

| Agent | Model tier | Job |
| --- | --- | --- |
| **Interviewer** | Flash-Lite | Works the idea over with structured, Claude-style multiple-choice-plus-free-text questions; drafts ahead by inference and fills outline sections as the conversation goes. The outline assembles **live in a canvas** beside the chat. |
| **Sufficiency judge** | Flash-Lite | Checks the outline against a rubric: a *deterministic* gate (all nine sections resolved) **plus** judged checks (no contradictions, happy path narratable end-to-end). Tells the user when the outline is adequate — but the user may always override in either direction. |
| **Feasibility gate** | Flash-Lite | Before offering a PoC, judges: would a static, single-file HTML mock *meaningfully* visualise this? Interfaces/dashboards → yes; headless pipelines/back-office automation → no, produce the flow map instead and say why. |
| **PoC generator** | Flash | Produces a self-contained single-file HTML proof-of-concept, with a visually distinct **limitations banner** authored *inside* the file (no real data, no real integrations, simulated logic). |
| **Flow-map generator** | Flash | Produces a Mermaid diagram of actors, systems, data stores and flows. The **SPA renders it to SVG in-browser** and posts the SVG back — Render's free tier can't run headless Chromium. |

The interview conversation is **unbounded**. The two-cycle revision cap applies only to
the PoC, the flow map, the threshold assessment, and the full assessment — *after* their
initial generation. A user can also **upload a file** instead of chatting (plain text
seeds the outline; a `.mmd` becomes the flow map; an `.html` becomes the PoC).

The user then explicitly **submits** to Governance. The outline is required; the PoC and
map are honestly encouraged but optional.

## 5. Phase 2 — Governance, part A: the threshold assessment (DTA §§1–4)

Governance begins in GitHub Actions. The threshold assessment answers: *is this a
low-risk idea that can conclude here, or does it need a full assessment?*

**Two generalist agents (Flash) draft sections 1–4 independently and in parallel.**
Independent parallel drafting was chosen over a sequential debate on purpose:
*disagreement between independent assessors is signal*, and the DTA's own guidance says
to resolve rating disagreement upward. Each generalist works from the brainstorm
artefacts and the DTA's own question text, guidance, consequence descriptors, and
likelihood table (Table 1).

**A reconciler agent (Pro) then merges the two drafts.** For each of the eight
inherent-risk categories in section 3, it adopts or synthesises each field, and where
the two generalists diverge on a consequence or likelihood tier it **takes the higher
tier** — recording the divergence and its reasoning in a visible *assessor divergence*
note (disagreement is surfaced, never hidden). Crucially, the reconciler reconciles the
**inputs** (consequence + likelihood), not the ratings.

**Then code — not any model — computes the ratings** (see §7). The completed threshold
assessment, including the deterministic §3.9 overall, is presented to the user at the
`THRESHOLD_REVIEW` pause. Routing follows the DTA's own logic:

- **all risks low** → the user *may* conclude here (`CONCLUDED`), the artefact framed as
  ready for an approving officer to consider;
- **any risk medium or high** → a full assessment is required;
- the user may also elect a full assessment regardless.

The threshold artefact is downloadable as markdown and supports the standard two
revision cycles.

## 6. Phase 2, part B: the full assessment — the specialist college (DTA §§5–12)

The full assessment is done by **six specialist agents**, each with clear, exclusive
ownership of a set of DTA sections and **its own private knowledge base**. A specialist
**cannot edit another specialist's work** — this isn't just an instruction, it's
*structural*: each agent emits JSON whose schema only permits keys for its owned
sections, and the assembler rejects anything out of scope.

| Specialist | Model | Owns (DTA sections) | Corpus docs | Indicative corpus |
| --- | --- | --- | --- | --- |
| **IT Security** | Flash | 6.7, 7.3 | 13 | ISM, OWASP ASVS & Agentic AI, ASD *Engaging with AI*, PSPF extracts |
| **Privacy** | Flash | 7.1, 7.2 | 8 | Privacy Act 1988 APPs, OAIC guidelines & PIA advice, Agencies Privacy Code, de-identification framework |
| **Ethics & Fairness** | Flash | 5.1, 5.2, 8.1, 8.2, 8.4, 8.5, 10.1 | 17 | AI Ethics Principles, NAIC guidance, CSIRO Responsible AI Pattern Catalogue, AIATSIS engagement |
| **Legal & Admin Law** | Flash | 9.1, 9.2, 10.2, 11.1, 12.1, 12.2 | 11 | Anti-discrimination Acts, Human Rights (Scrutiny) Act, Ombudsman ADM guide, ADJR overview, Robodebt RC context |
| **Data Governance** | Flash | 6.1, 6.2, 8.3 | 5 | APS Data Ethics Framework, ABS Data Quality Framework, Indigenous Data Governance (CARE/FAIR), NAA records advice |
| **Solution Architect** | Flash | 6.3–6.6, 6.8 (+ the Implementation Plan appendix) | 56 | DTA AI technical standard, AI procurement guidance, NIST AI RMF + Playbook, MIT AI Risk Repository |

Each specialist works from: the brainstorm artefacts, the completed threshold
assessment, **retrieval over its own KB**, and the DTA's question text + guidance for
its owned sections. Specialists have broad creative freedom (subheadings, diagrams,
worked examples) with one hard requirement: **every claim resting on the corpus cites
the document and a pinpoint locator** (a true page for PDFs; a provision, heading, or
sheet/row anchor otherwise). Where the scoping material is missing something, a
specialist does **not** silently guess — it states the gap, recommends the concrete next
step, and flags it so gaps aggregate into a *Recommended next steps* register.

**The question checkpoint (a single pause).** Each specialist may ask the user at most
**three** questions, and only where guessing would be genuinely inappropriate — the
happy path is *zero* questions. Mechanically it's one pause: all specialists draft, all
questions are batched into `questions.json`, the run pauses, and once answered each
specialist revises its own sections once. Skipped questions become flagged gaps.

**The Solution Architect wears two hats.** As a *specialist* it drafts its owned
sections (6.3–6.6, 6.8) in the six-way "bloom". Then, *after every specialist is
final*, it acts as the **architect** and writes the Implementation Plan appendix —
architecture, sequencing, diagrams, and explicit traceability back to the mitigations
the specialists proposed, so the plan demonstrably *answers* the assessment. Note that
**§12.5 (internal governance body review) is never done by an agent** — no AI can
perform it, so it is emitted as a flagged human action into the gap register.

## 7. How specialists read their corpus — retrieval and chunking

This is where Windtunnel deliberately departs from a conventional RAG stack, and the
reasoning matters.

### 7.1 One knowledge base per specialist

Each specialist has a single SQLite file, `kb/<specialist>.sqlite`, built from the
documents in `corpus/<specialist>/`. Alongside it sits a committed, human-readable
**index** (`.index.json`) and a **manifest** (`.manifest.json`) recording exactly which
documents, versions, and licences produced the KB.

### 7.2 Retrieval strategy — *index + fetch*, not vector similarity

Specialists do **survey-and-synthesis**, not needle-in-haystack lookup: given a novel
concept, each must judge *which of the things in its library apply*, then read them. So
instead of guessing an information need from an embedded query, retrieval hands the
model **the whole map of its library and lets it choose**. A specialist draft is *one
budgeted call containing a bounded tool loop*:

1. The system prompt embeds the specialist's **index** — a catalogue of everything its
   library contains (per document: what it is, plus a structure tree of sections/sheets
   with one-line descriptions, chunk-id ranges, and registry key ranges).
2. The wrapper **pre-fetches seed context** (a lexical BM25 search over the owned DTA
   question text + outline keywords) so grounding never starts empty.
3. The model calls two deterministic, **LLM-free** tools for up to `max_rounds` (default
   4) rounds:
   - **`fetch(refs)`** — refs are chunk ids, section paths, or **record keys** like
     `ISM-1612`, `APP 6`, or pattern `G4`; the code returns exactly those chunks.
   - **`search(query, k)`** — FTS5 BM25 lexical search, the backstop for anything the
     index descriptions under-sell.
4. Every returned chunk arrives as `(short_name, locator, text)`, every fetch/search
   emits a `retrieval` event and lands in provenance, and **the model can only cite what
   it actually fetched**. What it *chose* to read is itself part of the audit trail (and
   drives the transparency animation).

**Why no embeddings?** The corpus is ~106 documents of *structured reference material*
(numbered control registries, legislation with formal provisions, criteria
spreadsheets, chaptered guidance) that fits comfortably as an index inside the model's
context. At this scale and shape, an LLM-navigated structural index beats small-model
dense retrieval on recall and citation quality — and, decisively, an index miss is
*visible* to the model (the map is right in front of it) and recoverable inside the
loop, whereas a dense-retrieval miss is *silent*. Silent recall loss is the one failure
mode this product cannot afford. It's also operationally simpler: just SQLite + FTS5 +
one chunker, with no fusion weights or rerank thresholds to tune.

### 7.3 Chunking strategy — structure-anchored, with true locators

Ingestion (a GitHub Action over `corpus/**`) chunks **along each document's own
structure**, never with a blind sliding window:

- **Structure-aware extraction** first: PDFs via PyMuPDF (true pages + headings); DOCX
  via the style tree (heading styles for guidance, legislative styles for Acts' provision
  structure); XLSX normalised per sheet; MD by heading tree.
- **Chunk along structure** — section, provision, control, pattern, or sheet row-group —
  packing small siblings to ~400–900 tokens and splitting oversized sections only at
  paragraph boundaries. **A chunk never crosses a structural boundary**, so its locator
  is always unambiguous. There is *no overlap*: structure replaces it.
- **Atomic numbered items** (controls, criteria, patterns, APPs) become `record` chunks
  carrying a **`record_key`**, so `fetch("ISM-1997")` returns exactly the right rows.
- Every chunk carries a typed **locator** — the most precise *human-checkable* pointer
  its format supports: `p.112` for a PDF, `s 6(1)` / `Sch 1 APP 6` for legislation,
  `§Break glass accounts` for prose, `Controls!r412–r471` for a spreadsheet. Citations
  render as `(short_name, locator)` — e.g. `[ISM, p.112]`, `[Privacy Act 1988, s 6]`.

### 7.4 The licence hard gate

Because the repo is public, every chunk *republishes* source text. So ingestion reads a
`.meta.yml` sidecar for each document and **fails the build loudly** unless the document
is cleared as redistributable with an allow-listed licence (Commonwealth CC-BY, OWASP,
and similar). It is a gate, not a router: nothing downstream can be reached without
passing it.

## 8. The rating engine — *models argue, code computes*

This is the single most important integrity rule, and it is worth restating precisely:
**no AI agent ever emits a risk rating.** Agents output only a **consequence tier**, a
**likelihood tier**, and an evidenced **rationale**. A pure, LLM-free function then
computes the rating.

The engine's *data* is transcribed verbatim from the DTA tool v1.0 and lives in
`instrument/`, so there is exactly one place the authoritative matrix exists:

- `likelihood_table.json` — the Table 1 likelihood tiers
  (Rare → Unlikely → Possible → Likely → Almost certain).
- `consequence_table.json` — the consequence tiers
  (Insignificant → Minor → Moderate → Major → Severe).
- `risk_matrix.json` — **Table 2**: every (consequence × likelihood) → rating.

The engine is two small functions:

- `rating(consequence, likelihood)` → the Table 2 rating (raises on any off-vocabulary
  tier, so a stray label fails *loudly* rather than miscomputing silently);
- `overall_rating(ratings)` → the §3.9 / §12.4 **highest-wins** result.

> **Fidelity note.** The real DTA Table 2 is *not* the conventional 5×5 textbook matrix
> — it tops out at **High** (there is no "Very high" tier) and several cells differ. It
> was transcribed cell-by-cell from the source, and the rating-engine tests are
> hand-worked from the *actual* tool. A single wrong cell would be a fidelity failure,
> which is why this is the non-negotiable, most-tested part of the codebase.

The same engine is reused for the **residual** rating at §§12.3/12.4 (the reviewer
computes it on post-mitigation inputs). And it holds *under revision too*: a user's
revision can change the reasoning that leads to a consequence/likelihood judgement, but
it can never set a rating — the engine always recomputes. The computed number is
therefore *provably* an output of code from the resolved inputs, not an opinion.

## 9. The adjudicating reviewer

Once the specialists and the architect are done, a **reviewer agent (Pro)** audits the
assembled document for two things:

- **Coverage** — every DTA question is either substantively answered **or** explicitly
  present in the gap register with a reason and a recommended next step. This is a
  mechanical checklist walk generated from `instrument/questions.json`, so a missing
  question is a *deterministic* finding, not a judgement call.
- **Coherence** — no internal contradictions between sections, and consistency with the
  threshold assessment. Each finding cites the conflicting statements by section (and,
  where corpus-based, by citation) so a human can check the reviewer's own reasoning.

On finding a conflict, the reviewer decides **which specialist's position is less well
supported** and issues an **amend directive** to *that* specialist to fix *its own*
section — with written reasons. The specialist amends, and the reviewer re-runs.

- The loop is **capped at two cycles.** Conflicts still live after two cycles are **not
  forced into false agreement** — they are written into a *Points of unresolved
  disagreement* block, each as two well-argued positions the system chose not to
  reconcile. For a governance artefact, honest disagreement is more credible than
  manufactured consensus.
- Every ruling is preserved in `reviewer/cycle_N.json` and surfaced in provenance, so a
  human can **audit the audit**.

## 10. Assembly — the notebook and the report

The final artefact is a **Jupyter notebook in assembly-and-provenance format —
non-executable** (built programmatically with nbformat; nothing runs). An **nbconvert
HTML render** is produced alongside it as the shareable deliverable for SMEs. Both are
committed under the run id and downloadable. (This very document is written in the same
spirit.)

The notebook's cell plan mirrors the DTA instrument exactly:

1. **Title block** — project title, the *DRAFT — for SME review* mark, run code, date,
   and the standing disclaimer.
2. **Sections 1–4 (threshold)** — the section-3 category table with consequence,
   likelihood, the **computed** rating chip, expandable rationales, the §3.9 overall
   (driver named), and the assessor divergence notes shown as rigour.
3. **Sections 5–12 (full)** — each specialist's owned sections in tool order, with
   inline `(short_name, locator)` citations resolved against the manifests.
4. **12.3 / 12.4** — residual risk summary and rating (engine reused);
   **12.5** as a flagged human action.
5. **Appendices** — Implementation Plan (architect); *Recommended next steps* (the
   aggregated gap register, including skipped questions); assessor divergence notes;
   *Points of unresolved disagreement*; and the **provenance** cell.
6. **Reference list** — the full pinpoint-cited apparatus, deduplicated across
   specialists.

A few deliberate choices: the **PoC is embedded** into the HTML as a sandboxed iframe
(self-contained); **all diagrams are pre-rendered SVG** because Mermaid does not render
in nbconvert (the `.mmd` source is kept only for provenance); and the **provenance
cell** records the run id, per-role model versions, prompt versions, corpus manifest
versions, agent-to-section attribution, token/cost totals, and the input attestation (a
self-attested confirmation that inputs contained no sensitive, classified, or personal
information, appropriate for a public repository — not a claim about system sensitivity
accreditation) — everything needed to reproduce and audit the result.

## 11. The transparency layer — the useful loading screen

Throughout Governance the user watches an **animated pipeline diagram** showing which
agents are active and what they're doing — retrievals ("Privacy specialist retrieving:
OAIC PIA guidance, p.14"), drafting, review cycles. It's a *trust instrument, not
decoration*: the goal is that a non-expert finishes a run confident that every element
was comprehensively considered.

The mechanism is simple and robust, and it reuses everything above:

- The animation is **pre-scripted against the fixed pipeline topology** — a known set of
  node ids shared verbatim between the pipeline and the UI (the two generalists, the
  reconciler, the rating engine, the six specialists + their knowledge-base nodes, the
  checkpoint, the architect, the reviewer, and assembly).
- The pipeline writes `status.json` on every transition: the whole-graph `nodes` map, an
  append-only event log with a **small controlled vocabulary** (`stage_started`,
  `retrieval`, `drafting`, `question_raised`, `revision`, `review_finding`,
  `stage_complete`, `heartbeat`, `error`), and a heartbeat at least every ~20 s.
- The SPA **polls `status.json` every few seconds** (through the Render proxy) and
  animates only genuinely new events. Near-real-time is the agreed fidelity — no
  websockets, no true streaming.
- A design rule keeps it honest: **no event may exist only in the graph** — every node
  state change is also stated in words in the log, which is the accessibility and
  honesty backbone. State is always carried by label + shape + position, never colour
  alone.

Because one poll fully determines the visible state, the animation is correct even if
the browser misses polls, sleeps, or reconnects mid-run.

## 12. Resilience, resume, and revision — the recurring pattern

A few rules recur across every phase, and together they make the system durable:

- **Run codes + checkpoints = resume.** Every run has a copyable code; the pipeline
  commits state after every stage. Paste the code later to resume from the last
  checkpoint. A paused run and a failed run resume by the *same* mechanism — a fresh
  dispatch reading `run.json`.
- **Idempotence is the resume model.** Any stage checks whether its checkpoint outputs
  already exist before redoing work, so it's always safe to run twice.
- **Revision caps.** The initial brainstorm conversation is unbounded; after initial
  generation, the flow map, the PoC, the threshold assessment, and the full assessment
  each allow **≤2** user-driven revisions. The reviewer's internal correction loop is
  separately capped at 2 cycles.
- **Graceful failure.** Any unhandled error becomes a calm, resumable failure state: a
  plain message for the user, the technical detail tucked behind "Show technical
  detail", and the last good checkpoint intact.

---

### Where to go deeper

| You want the detail on… | Read |
| --- | --- |
| *Why* the product is shaped this way; the fixed decisions | `PROJECT_BRIEF.md` |
| The pipeline, data contracts, KBs, retrieval, the state machine | `TECH_SPEC.md` (esp. §5, §6, §8, §10) |
| The interface, the transparency animation, the report styling | `DESIGN_BRIEF.md` |
| The DTA instrument itself (Tables 1 & 2, consequence appendix) | `instrument/guidance/` |
| What's built, what's in flight, decisions taken | `STATUS.md` |
| Conventions and invariants for anyone editing the repo | `CLAUDE.md` |

---

*This overview is a map, not the territory. The documents above are authoritative where
they and this notebook ever disagree: the project brief governs intent, the design brief
governs the interface and the report, and the tech spec governs the pipeline, the data
contracts, and the build.*